# 01 — The dataset, and what hashing it revealed

1 506 MRI files: a labelled pool of 100 (normal / cancer) and an unlabelled pool of 1 406.
The plan was to embed them with a frozen ResNet50, cluster the embeddings, and turn the
clusters into pseudo-labels for a semi-supervised model.

This notebook starts one step earlier, with a question the original pipeline never asked:
**are these 1 506 files 1 506 images?**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Identity comes from content, not from the path

The first version of this pipeline identified an image by an MD5 of its **file path**. Two
copies of one scan filed in two folders therefore received two different identifiers — and
the guard meant to keep the labelled images out of the unlabelled pool, an exclusion on the
folder name, had nothing to catch them with.

Here an image is what it contains.

In [ ]:
from mri_semisupervised.config import LABELED_DIR, UNLABELED_DIR
from mri_semisupervised.data.manifest import apply_duplicate_rules, build_manifest, summarise

manifest = apply_duplicate_rules(build_manifest(LABELED_DIR, UNLABELED_DIR))
facts = summarise(manifest)
for key, value in facts.items():
    print(f"{key:34} {value}")

## 2. What the hashes found

Three findings, none of them visible from the folder listing:

* the labelled pool holds **one image twice**, so the evaluation set is 99 images, not 100 —
  and the class balance is 50 / 49, not the announced 50 / 50;
* **31 of those 99 evaluation images also sit in the unlabelled pool**, byte for byte. The
  semi-supervised arm pre-trains on that pool, so a third of every test fold was already
  seen — whatever the cross-validation does;
* the unlabelled pool carries 63 redundant copies, which silently weight the pre-training.

The leak is also **asymmetric**: 23 `normal` against 8 `cancer`. It does not add noise, it
leans.

In [ ]:
leaked_ids = set(
    manifest.loc[
        (manifest["pool"] == "unlabelled") & (manifest["exclusion_reason"] == "copy_of_labelled"),
        "image_id",
    ]
)
leaked = manifest[(manifest["pool"] == "labelled") & manifest["image_id"].isin(leaked_ids)]
print(f"evaluation images with a copy in the unlabelled pool: {leaked['image_id'].nunique()}")
leaked.drop_duplicates("image_id")["label"].value_counts().rename("images").to_frame()

### The rules, written down because they are decisions

| situation | decision | why |
|---|---|---|
| labelled image duplicated in the unlabelled pool | leaves **that pool** | removing it from the evaluation would be choosing the test set after seeing it |
| redundant copies inside the unlabelled pool | one kept | otherwise they weight the pre-training without saying so |
| an image repeated inside the labelled pool | one kept | kept twice, it sits on both sides of a fold |
| the same content under two labels | **build stops** | that is not a duplicate, it is a contradiction |

In [ ]:
manifest["exclusion_reason"].fillna("kept").value_counts().rename("files").to_frame()

## 3. What the images look like

Everything is 512x512 RGB. The two classes differ in intensity distribution, which is worth
knowing before reading anything into a clustering that separates them.

In [ ]:
from mri_semisupervised.data.loader import compute_pixel_stats, discover_images
from mri_semisupervised.viz.plots import plot_image_grid, plot_pixel_stats

records, corrupted = discover_images()
print(f"{len(records)} readable files, {len(corrupted)} unreadable")

stats = compute_pixel_stats(records, sample_size=300, seed=0)
plot_pixel_stats(stats)

In [ ]:
labelled = manifest[(manifest["pool"] == "labelled") & manifest["kept_for_training"]]
sample = labelled.groupby("label").head(4)
plot_image_grid(sample["path"].tolist(), titles=sample["label"].tolist(), cols=4)

### Histogram equalisation

Stretching the intensities makes structure easier to see. It is used here for looking, not
for the features: the backbone was trained under ImageNet normalisation, and feeding it
something else would trade a small visual gain for a distribution shift.

In [ ]:
from mri_semisupervised.viz.plots import plot_equalization_demo

plot_equalization_demo(sample["path"].iloc[0])

## 4. Clustering the embeddings

Five algorithms on the ResNet50 embeddings, compared on internal metrics and on the ARI
against the labels.

**Where the ARI is computed matters.** In the corrected protocol it is computed on the
*training fold's* labels only, inside the fold. Here, in exploration, it is computed on all
of them — which is legitimate for looking at the data, and was exactly the mistake when the
same number was used to *choose* the method that produced the pseudo-labels.

In [ ]:
from mri_semisupervised.config import ClusteringConfig, FeatureConfig
from mri_semisupervised.features.extractor import load_cached_features
from mri_semisupervised.models.clustering import (
    build_clustering_report,
    fit_agglomerative,
    fit_dbscan,
    fit_gmm,
    fit_kmeans,
    reduce_pca,
    standardise,
)

features, index_df = load_cached_features(FeatureConfig().cache_path)
truth = index_df["label_index"].to_numpy(dtype=float)

reduced, _ = standardise(features)
reduced, _ = reduce_pca(reduced, target_variance=ClusteringConfig().pca_variance)
print(f"{features.shape[1]} dimensions -> {reduced.shape[1]} components at 95% variance")

results = [
    fit_kmeans(reduced, truth, n_clusters=2),
    fit_agglomerative(reduced, truth, n_clusters=2, linkage="ward"),
    fit_agglomerative(reduced, truth, n_clusters=2, linkage="average"),
    fit_gmm(reduced, truth, n_components=2),
    fit_dbscan(reduced, truth, eps=8.0, min_samples=10),
]
build_clustering_report(results)

### What the table says, and what it does not

Read the silhouette column beside the ARI column. `Agglomerative(average)` has the best
silhouette of the five and an ARI of zero: it found a very tight structure that has nothing
to do with the diagnosis — one cluster holding almost everything, and one holding a handful
of outliers.

An internal metric measures whether a partition is *neat*. It cannot tell you whether it is
*the one you wanted*. That is the whole reason the ARI is in the table.

DBSCAN, at this eps, declares nearly everything noise. Reported rather than tuned away: a
method that refuses to split is information about the embedding space.

In [ ]:
from mri_semisupervised.viz.plots import plot_2d_scatter, project_2d

best = max((r for r in results if r.ari_vs_truth is not None), key=lambda r: r.ari_vs_truth)
print(f"best ARI against the labels: {best.name} ({best.ari_vs_truth:.3f})")

coords = project_2d(reduced, method="tsne", seed=42)
labels = index_df["label_name"].fillna("unlabeled").tolist()
plot_2d_scatter(coords, labels, title="t-SNE of the ResNet50 embeddings, coloured by label")

## 5. Where this leaves the pseudo-labels

The best clustering reaches an ARI around 0.6 against the true labels. That is substantial
and far from decisive: the pseudo-labels it produces will be right often enough to be worth
trying, and wrong often enough that the trying has to be measured rather than assumed.

Notebook 02 measures it — under a protocol where the test fold takes part in no decision,
and against a control that receives the same images with their pseudo-labels shuffled.